# Sprint 4 — RAG com página, seção e estado operacional

**Challenge FIAP — Forzy | Processamento de Linguagem Natural**

Este notebook compara o preset legado com uma recuperação em português que preserva a origem de cada trecho e usa alertas ativos para orientar a busca.

| Seção | Conteúdo |
|---|---|
| 0 | Objetivo e roteiro |
| 1 | Setup local e chave do OpenRouter |
| 2 | Extração por página, chunking e comparação dos presets |
| 3 | Embeddings, reranker e indexação |
| 4 | Estado operacional a partir das leituras reais |
| 5 | Busca com e sem estado |
| 6 | Conclusões e limites |
| 7 | Assistente de troubleshooting |
| 8 | Smoke do assistente |
| 9 | Gabarito de avaliação (20 dentro do escopo + 5 fora) |
| 10 | Recuperação: LEGADO × NOVO sem estado × NOVO com estado |
| 11 | Juiz por afirmação: protocolo e calibração |
| 12 | Respostas: faithfulness, relevancy e abstenção |
| 13 | Análise de erros |
| 14 | Como os cenários são escolhidos |
| 15 | Cenário 1 — anomalia elétrica |
| 16 | Cenário 2 — anomalia mecânica |
| 17 | Cenário 3 — manutenção preventiva |
| 18 | Limites e mitigações (cenário adversarial) |


## 1. Setup

Execute a partir da raiz do repositório. O bloco localiza a raiz sem clonar o projeto nem instalar pacotes; as dependências ficam em `requirements.txt`. A chave é lida do `.env` da raiz e nunca exibida.


In [1]:
from pathlib import Path
import sys

candidatos_raiz = (Path.cwd().resolve(), *Path.cwd().resolve().parents)
ROOT = next((candidato for candidato in candidatos_raiz if (candidato / "src" / "pln" / "config.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Abra o notebook dentro do repositório que contém src/pln/config.py.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.pln.config import NOVO, LEGADO, PDF_CATALOGO_FILE
from src.pln.llm import load_openrouter_api_key

try:
    chave_openrouter_disponivel = bool(load_openrouter_api_key(ROOT / ".env"))
except RuntimeError:
    chave_openrouter_disponivel = False

print(f"Raiz do repositório: {ROOT}")
print(f"PDF do catálogo encontrado: {PDF_CATALOGO_FILE.is_file()}")
print(f"Chave OpenRouter disponível: {chave_openrouter_disponivel}")


Raiz do repositório: C:\Users\Igor\Desktop\Sarak\Fiap\CP - Sprint - GS\2º Ano\Sprints\Sprint 4
PDF do catálogo encontrado: True
Chave OpenRouter disponível: True


## 2. Extração e chunking

A extração remove os elementos repetidos do cabeçalho/rodapé, mantém o número impresso, identifica seções e converte tabelas em linhas rotuladas. O preset `LEGADO` mantém o agrupamento por parágrafo sem metadados; `NOVO` mantém página e seção.


In [2]:
from src.pln.config import LEGADO, NOVO, PDF_CATALOGO_FILE
from src.pln.rag import chunk_legado, chunk_novo, extrair_paginas

paginas = extrair_paginas(PDF_CATALOGO_FILE)
chunks_legado = chunk_legado("\n\n".join(pagina.texto_original for pagina in paginas), LEGADO)
chunks_novos = chunk_novo(paginas, NOVO)

print(f"Páginas extraídas: {len(paginas)}")
print(f"Chunks LEGADO: {len(chunks_legado)} | limite: {LEGADO.chunk_max} caracteres")
print(f"Chunks NOVO: {len(chunks_novos)} | limite: {NOVO.chunk_max} caracteres")
print(f"Chunks NOVO com página e seção: {sum(chunk.pagina is not None and bool(chunk.secao) for chunk in chunks_novos)}")
for chunk in chunks_novos[:3]:
    print(f"\nPágina {chunk.pagina} — seção {chunk.secao} — {chunk.secao_titulo}\n{chunk.texto}")


Consider using the pymupdf_layout package for a greatly improved page layout analysis.


Páginas extraídas: 52
Chunks LEGADO: 77 | limite: 1000 caracteres
Chunks NOVO: 699 | limite: 900 caracteres
Chunks NOVO com página e seção: 699

Página 1 — seção 0 — Conteúdo sem seção numerada
[Seção 0 — Conteúdo sem seção numerada | p. 1-3] Motores

Automação

Energia

Transmissão &

Distribuição

Tintas

W22

Motor Elétrico Trifásico

Catálogo Técnico Mercado Brasil

Driving efﬁciency and sustainability

W22

Muito mais eficiência, qualidade

e economia para a sua indústria.

Carcaças 63 a 132

Carcaças 160 a 200

Carcaças 225 a 355

Eficiência e Confiabilidade para a Indústria

Neste catálogo técnico são apresentadas as versões de

motores elétricos de indução IR3 Premium e Super Premium.

A linha W22 da WEG está disponível em três versões,

W22 IR3 Premium, W22 Super Premium (motor elétrico

de indução) e W22 Ultra Premium (motor elétrico de ímãs

permanentes).

Estes motores têm perdas entre 10% e 40% menores que

os anteriormente disponíveis no mercado. É uma maneira

extremamen

## 3. Indexação

O índice usa ChromaDB efêmero com distância cosseno. O preset `NOVO` aplica embeddings E5 multilíngues e reranking multilíngue. O tempo abaixo inclui o carregamento dos modelos na primeira execução e a criação do índice.


In [3]:
from time import perf_counter

from src.pln.rag import criar_retriever

inicio_indexacao = perf_counter()
retriever = criar_retriever(chunks_novos, NOVO)
tempo_indexacao = perf_counter() - inicio_indexacao
print(f"Embedding: {NOVO.modelo_embedding}")
print(f"Reranker: {NOVO.modelo_reranker}")
print(f"Passagens indexadas: {len(chunks_novos)}")
print(f"Tempo de carregamento e indexação: {tempo_indexacao:.1f} s")


C:\Users\Igor\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2600.26it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 2406.27it/s]

Embedding: intfloat/multilingual-e5-base
Reranker: cross-encoder/mmarco-mMiniLMv2-L12-H384-v1
Passagens indexadas: 699
Tempo de carregamento e indexação: 332.1 s


## 4. Estado operacional

Os episódios são derivados das leituras reais de sensores. A primeira amostra usa o início de um alerta crítico; a segunda usa uma leitura real fora de qualquer janela de alerta.


In [4]:
from src.pln.alertas import carregar_leituras, detectar_episodios
from src.pln.estado import construir_estado_operacional

leituras = carregar_leituras()
alertas = detectar_episodios(leituras)
alerta_critico = next(alerta for alerta in alertas if alerta.severidade == "critico")
instante_critico = alerta_critico.inicio
instantes_leitura = sorted((leitura["timestamp"] for leitura in leituras), reverse=True)
instante_normal = next(
    instante
    for instante in instantes_leitura
    if not any(alerta.inicio <= instante <= alerta.fim for alerta in alertas)
)
estado_critico = construir_estado_operacional(alertas, instante_critico)
estado_normal = construir_estado_operacional(alertas, instante_normal)

print(f"Episódios derivados: {len(alertas)}")
print(f"Instante crítico: {estado_critico.texto()}")
print(f"Instante normal: {estado_normal.texto()}")


Episódios derivados: 4477
Instante crítico: Teste (Sorocaba): severidade geral critico. Alerta A-0001: Temperatura do enrolamento (estator), severidade critico, valor máximo 122.96 °C, início 2026-05-05T01:22:22+00:00.
Instante normal: Teste (Sorocaba): operação normal em 2026-08-23T09:46:39+00:00.


## 5. Retriever com e sem estado

A mesma pergunta percorre o retriever em duas condições. Sem estado, não há expansão por termos operacionais nem bônus de seção. Com estado, a busca inclui os termos do domínio ativo e pode receber o bônus definido no preset.


In [5]:
pergunta = "Quais verificações devo fazer para entender o aquecimento do motor?"

resultados_sem_estado = retriever.buscar(pergunta, estado=None)
resultados_com_estado = retriever.buscar(pergunta, estado=estado_critico)

def mostrar_resultados(rotulo, resultados):
    print(f"\n{rotulo}")
    for posicao, resultado in enumerate(resultados, start=1):
        print(
            f"{posicao}. p. {resultado['pagina']} — seção {resultado['secao']} — "
            f"{resultado['secao_titulo']} | denso={resultado['score_denso']:.3f} "
            f"rerank={resultado['score_rerank']:.3f} final={resultado['score_final']:.3f}"
        )
        print(resultado["texto"][:320])

mostrar_resultados("Sem estado", resultados_sem_estado)
mostrar_resultados("Com estado crítico", resultados_com_estado)



Sem estado
1. p. 19 — seção 10 — Ambiente x Isolamento | denso=0.658 rerank=1.000 final=1.000
[Seção 10 — Ambiente x Isolamento | p. 19] Deve-se observar a potência nominal indicada nas tabelas

elétricas, exceto quando especificado de maneira diferente à

operação em regime contínuo S1, conforme ABNT 17094 em

ambientes:

gCom temperatura variando entre -30 °C a +40 °C.

gEm altitudes de até 1000 metros acima 
2. p. 20 — seção 11 — Proteção Térmica do Motor | denso=0.000 rerank=0.875 final=0.875
[Seção 11 — Proteção Térmica do Motor | p. 20-21] Figura 30 - Termistor (PTC)

As proteções disponíveis para o motor podem ser

classificadas em:

g Dependência da temperatura de funcionamento.

g Dependência da corrente de funcionamento.

Embora não permitam um acompanhamento contínuo do

processo de aquecimento do mo
3. p. 20 — seção 10.1 — Resistências de Aquecimento | denso=0.281 rerank=0.875 final=0.875
[Seção 10.1 — Resistências de Aquecimento | p. 20] assim um acompanhamento contínuo d

## 6. Conclusão

O preset `NOVO` mantém página e seção nos chunks, serializa tabelas em linhas legíveis e expande a busca com termos associados aos alertas ativos. `LEGADO` permanece disponível para a comparação planejada.

A seção 5 permite observar a mudança de consulta e de ordenação com o mesmo enunciado. O resultado depende dos modelos multilíngues configurados e do catálogo W22; não deve ser interpretado como avaliação de qualidade sem o conjunto de perguntas e métricas da próxima etapa.

**Execução local:** rode as células em ordem para baixar/carregar os modelos e verificar os resultados reais. Não salve outputs ou chaves no notebook.


## 7. Assistente de troubleshooting

O assistente usa o estado operacional como dado confiável e os chunks recuperados como única fonte técnica. O score bruto do reranker determina abstenção e confiança; página e seção dos chunks validam as citações. A memória mantém até `MEMORIA_TURNOS` turnos e amplia consultas curtas ou anafóricas com a pergunta anterior.

### Inspeção offline e ponto de partida do limiar

Inspecionei 10 consultas dentro do escopo — incluindo duas buscas já compostas com a pergunta anterior — e 6 consultas fora dele. Os valores abaixo são `score_rerank_bruto` do primeiro chunk retornado pelo retriever, usando o mesmo estado crítico passado ao smoke. A regra de partida é somar a margem mínima ao maior score fora do escopo e arredondar o resultado para cima ao décimo: `ceil((maior_score_fora + margem) * 10) / 10`.

O maior score fora do escopo foi -3.027231. Com margem mínima de 0.1, o valor intermediário é -2.927231 e o arredondamento para cima resulta em -2.9, o ponto de partida desta inspeção. Nesta amostra pequena nenhum limiar separa as duas populações: o menor score dentro do escopo (-4.751304, seguimento de ventilação) é menor que o maior score fora dele (-3.027231). O valor vigente de `LIMIAR_ABSTENCAO` (2.0) não vem desta inspeção: foi calibrado no conjunto `dev` do gabarito, na seção 10.

| Escopo | Consulta usada na medição | Melhor score bruto | Página/seção do primeiro chunk |
|---|---|---:|---|
| Dentro | Qual é o grau de proteção do motor WEG W22? | 9.925054 | 18 / 7 |
| Dentro | Qual é o grau de proteção do motor WEG W22? E o que essa proteção significa? | 8.729171 | 18 / 7 |
| Dentro | Qual é a classe de isolamento do motor WEG W22? | 6.704764 | 20 / 11.1 |
| Dentro | Quais são as características do sistema de ventilação? | -4.239105 | 21 / 12.1 |
| Dentro | Quais são as características do sistema de ventilação? E quais são as características desse sistema? | -4.751304 | 21 / 12.1 |
| Dentro | Quais dados técnicos aparecem na placa de identificação do motor WEG W22? | 9.336778 | 12 / 3.11 |
| Dentro | Qual é a faixa de temperatura ambiente permitida para o motor WEG W22? | 7.954395 | 19 / 9 |
| Dentro | Qual altitude de instalação é permitida para o motor WEG W22? | 6.590040 | 20 / 11.1 |
| Dentro | Como deve ser feita a lubrificação dos rolamentos do motor WEG W22? | 4.264139 | 14 / 5.1 |
| Dentro | Quais são os limites de vibração permitidos para o motor WEG W22? | 7.368129 | 14 / 5.2 |
| Fora | Como assar pão de queijo? | -6.147628 | 20 / 11.1 |
| Fora | Qual é a população atual do Brasil? | -3.027231 | 38 / 16 |
| Fora | Como configurar uma impressora a laser? | -6.134845 | 33 / 15 |
| Fora | Qual é a capital da França? | -3.982897 | 38 / 16 |
| Fora | Como preparar um bolo de chocolate? | -6.812472 | 38 / 16 |
| Fora | Como configurar um roteador Wi-Fi doméstico? | -5.741543 | 33 / 15 |


In [6]:
from IPython.display import Markdown, display
from src.pln.config import (
    FAIXAS_CONFIANCA,
    LIMIAR_ABSTENCAO,
    MARGEM_ABSTENCAO,
    LIMIAR_PALAVRAS_CONSULTA,
    MEMORIA_TURNOS,
    MODELO_GERADOR,
    PROMPT_SISTEMA_ASSISTENTE,
)

display(Markdown(f"```text\n{PROMPT_SISTEMA_ASSISTENTE}\n```"))
display(Markdown(
    "|Parâmetro|Valor|\n|---|---|\n"
    f"|Modelo gerador|`{MODELO_GERADOR}`|\n"
    f"|Memória em turnos|`{MEMORIA_TURNOS}`|\n"
    f"|Limiar de consulta curta|`{LIMIAR_PALAVRAS_CONSULTA}`|\n"
    f"|Limiar de abstenção|`{LIMIAR_ABSTENCAO}`|\n"
    f"|Margem mínima do limiar|`{MARGEM_ABSTENCAO}`|\n"
    f"|Faixas de confiança|`{FAIXAS_CONFIANCA}`|"
))


```text
Você é um assistente técnico especialista em motores elétricos WEG W22. Responda somente com informações presentes no bloco CONTEXTO DO MANUAL. Cite cada afirmação factual usando [p. <página>, seç. <seção>] e use apenas páginas e seções presentes no contexto. Não use conhecimento externo e não invente números. Trate o bloco ESTADO OPERACIONAL como dado confiável do sistema; não reinterprete nem conteste seus valores ou severidades. Se o contexto do manual não sustentar a resposta, responda exatamente: Não encontrei informações suficientes no manual do motor WEG W22 para responder a essa pergunta. Termine com uma linha Confiança: <Alta|Média|Baixa>.
```

|Parâmetro|Valor|
|---|---|
|Modelo gerador|`meta-llama/llama-3.3-70b-instruct`|
|Memória em turnos|`3`|
|Limiar de consulta curta|`6`|
|Limiar de abstenção|`2.0`|
|Margem mínima do limiar|`0.1`|
|Faixas de confiança|`{'Alta': {'score_minimo': 8.0, 'chunks_minimos': 3}, 'Média': {'score_minimo': 0.0, 'chunks_minimos': 1}}`|

## 8. Smoke do assistente

Execute as três perguntas dentro do escopo, os dois seguimentos de memória e as três perguntas fora do escopo. Cada resultado mostra a consulta usada, fontes, confiança, abstenção e modelo. Para respostas não abstidas, o smoke também imprime o texto completo, incluindo citações e a linha de confiança. Esta seção faz chamadas ao gerador e depende da chave local do OpenRouter.


In [7]:
from src.pln.assistente import Assistente

perguntas_smoke = (
    "Qual é o grau de proteção do motor WEG W22?",
    "E o que essa proteção significa?",
    "Qual é a classe de isolamento do motor WEG W22?",
    "Quais são as características do sistema de ventilação?",
    "E quais são as características desse sistema?",
    "Como assar pão de queijo?",
    "Qual é a população atual do Brasil?",
    "Como configurar uma impressora a laser?",
)
assistente_demo = Assistente(retriever)
respostas_smoke = []
for pergunta_smoke in perguntas_smoke:
    resposta_smoke = assistente_demo.responder(pergunta_smoke, estado=estado_critico)
    respostas_smoke.append(resposta_smoke)
    print(
        {
            "pergunta": pergunta_smoke,
            "consulta_usada": resposta_smoke.consulta_usada,
            "fontes": resposta_smoke.fontes,
            "confianca": resposta_smoke.confianca,
            "abstencao": resposta_smoke.abstencao,
            "modelo": resposta_smoke.modelo,
        }
    )
    if not resposta_smoke.abstencao:
        print(resposta_smoke.texto)


{'pergunta': 'Qual é o grau de proteção do motor WEG W22?', 'consulta_usada': 'Qual é o grau de proteção do motor WEG W22?', 'fontes': [{'pagina': 18, 'secao': '7', 'secao_titulo': 'Grau de proteção / Vedação / Pintura', 'chunk_id': 'novo-00177'}], 'confianca': 'Alta', 'abstencao': False, 'modelo': 'meta-llama/llama-3.3-70b-instruct'}
O grau de proteção dos motores W22 varia de acordo com o tamanho da carcaça. Para carcaças de 63 a 132M/L, o grau de proteção é IP55, e para carcaças de 160M a 355A/B, o grau de proteção é IPW55 [p. 18, seç. 7]. 
Confiança: Alta


{'pergunta': 'E o que essa proteção significa?', 'consulta_usada': 'Qual é o grau de proteção do motor WEG W22? E o que essa proteção significa?', 'fontes': [{'pagina': 18, 'secao': '7', 'secao_titulo': 'Grau de proteção / Vedação / Pintura', 'chunk_id': 'novo-00177'}, {'pagina': 14, 'secao': '4.4', 'secao_titulo': 'Resistência ao Impacto', 'chunk_id': 'novo-00053'}], 'confianca': 'Alta', 'abstencao': False, 'modelo': 'meta-llama/llama-3.3-70b-instruct'}
A proteção IP55, presente nas carcaças 63 a 132M/L, significa que o invólucro fornece proteção contra o contato ou aproximação com partes vivas ou móveis dentro do invólucro e contra o ingresso de poeira e jatos de água [p. 18, seç. 7]. Já a proteção IPW55, presente nas carcaças 160M a 355A/B, oferece uma proteção semelhante, mas com especificações adicionais para ambientes mais agressivos. Além disso, os motores W22 também atendem ao índice de impacto IK08, o que garante elevada rigidez mecânica em aplicações mais severas [p. 14, seç.

{'pergunta': 'Qual é a classe de isolamento do motor WEG W22?', 'consulta_usada': 'Qual é a classe de isolamento do motor WEG W22?', 'fontes': [{'pagina': 19, 'secao': '8', 'secao_titulo': 'Tensão / Frequência', 'chunk_id': 'novo-00187'}], 'confianca': 'Média', 'abstencao': False, 'modelo': 'meta-llama/llama-3.3-70b-instruct'}
A classe de isolamento dos motores W22 é F [p. 19, seç. 8].
Confiança: Média


{'pergunta': 'Quais são as características do sistema de ventilação?', 'consulta_usada': 'Quais são as características do sistema de ventilação?', 'fontes': [], 'confianca': 'Baixa', 'abstencao': True, 'modelo': 'meta-llama/llama-3.3-70b-instruct'}


{'pergunta': 'E quais são as características desse sistema?', 'consulta_usada': 'Quais são as características do sistema de ventilação? E quais são as características desse sistema?', 'fontes': [], 'confianca': 'Baixa', 'abstencao': True, 'modelo': 'meta-llama/llama-3.3-70b-instruct'}


{'pergunta': 'Como assar pão de queijo?', 'consulta_usada': 'E quais são as características desse sistema? Como assar pão de queijo?', 'fontes': [], 'confianca': 'Baixa', 'abstencao': True, 'modelo': 'meta-llama/llama-3.3-70b-instruct'}


{'pergunta': 'Qual é a população atual do Brasil?', 'consulta_usada': 'Qual é a população atual do Brasil?', 'fontes': [], 'confianca': 'Baixa', 'abstencao': True, 'modelo': 'meta-llama/llama-3.3-70b-instruct'}


{'pergunta': 'Como configurar uma impressora a laser?', 'consulta_usada': 'Como configurar uma impressora a laser?', 'fontes': [], 'confianca': 'Baixa', 'abstencao': True, 'modelo': 'meta-llama/llama-3.3-70b-instruct'}


## 9. Gabarito de avaliação

O gabarito (`data/avaliacao_rag.json`) tem 20 perguntas **dentro do escopo**, cada uma com resposta de referência, página(s) e seção(ões) do PDF e um `trecho_ancora`, frase literal do catálogo que sustenta a resposta; e 5 perguntas **fora do escopo**, cuja resposta correta é a abstenção. Cinco das 20 formam o `dev` (usado para calibrar parâmetros) e 15 formam o `teste`. Nove perguntas dentro do escopo trazem um cenário de estado operacional, construído a partir de alertas reais: o estado no início do primeiro alerta crítico que envolve só aquela grandeza. Duas perguntas fora do escopo também recebem estado, para conferir que ele não derruba a abstenção.

O teste da âncora abaixo falha se alguma frase do gabarito não existir na página indicada, e se o termo-chave de uma pergunta fora do escopo existir no PDF.

In [8]:
from collections import Counter

from src.pln.avaliacao import ancora_presente, carregar_gabarito, construir_estados, termo_ausente_no_pdf, textos_por_pagina

perguntas = carregar_gabarito()
dentro = [pergunta for pergunta in perguntas if pergunta.escopo == "dentro"]
fora = [pergunta for pergunta in perguntas if pergunta.escopo == "fora"]
print(f"Dentro do escopo: {len(dentro)} | fora do escopo: {len(fora)}")
print("Split (dentro):", dict(Counter(pergunta.split for pergunta in dentro)))
print("Categorias (dentro):", dict(Counter(pergunta.categoria for pergunta in dentro)))
estados_usados = Counter(pergunta.estado for pergunta in dentro if pergunta.estado)
print("Perguntas dentro do escopo com estado:", sum(estados_usados.values()), dict(estados_usados))

for identificador in ("e02", "m05", "f01"):
    exemplo = next(pergunta for pergunta in perguntas if pergunta.id == identificador)
    print(f"\n[{exemplo.id}] {exemplo.pergunta}")
    if exemplo.escopo == "dentro":
        print(f"  referência: {exemplo.resposta_referencia}")
        print(f"  páginas {list(exemplo.paginas_gabarito)} | seções {list(exemplo.secoes_gabarito)} | {exemplo.categoria} | {exemplo.split}")
        print(f"  âncora: \"{exemplo.trecho_ancora}\"")
    else:
        print(f"  esperado: {exemplo.esperado} (o termo \"{exemplo.termo_ausente}\" não ocorre no PDF)")

textos_pdf = textos_por_pagina(paginas)
sem_ancora = [pergunta.id for pergunta in dentro if not ancora_presente(pergunta, textos_pdf)]
termo_presente = [pergunta.id for pergunta in fora if not termo_ausente_no_pdf(pergunta, textos_pdf)]
print(f"\nPerguntas cuja âncora não está na página do gabarito: {sem_ancora}")
print(f"Perguntas fora do escopo cujo termo-chave existe no PDF: {termo_presente}")
assert not sem_ancora and not termo_presente

estados = construir_estados(alertas)
for chave, estado_cenario in estados.items():
    print(f"\n{chave}: {estado_cenario.texto()}")

Dentro do escopo: 20 | fora do escopo: 5
Split (dentro): {'teste': 15, 'dev': 5}
Categorias (dentro): {'elétrica': 5, 'mecânica': 5, 'preventiva': 5, 'especificação': 5}
Perguntas dentro do escopo com estado: 9 {'temperatura_critico': 4, 'corrente_critico': 2, 'vibracao_critico': 3}

[e02] Que corrente o motor W22 deve suportar durante 2 minutos segundo a norma ABNT NBR 17094?
  referência: Motores de potência nominal até 315 kW e tensão nominal de até 1 kV devem suportar uma corrente de 1,5 vez a corrente nominal durante 2 minutos, no mínimo.
  páginas [19] | seções ['9'] | elétrica | dev
  âncora: "devem ser capazes de suportar uma corrente igual a 1,5 vez a corrente nominal durante 2 minutos, no mínimo"

[m05] Como a vedação WSeal impede a entrada de poeira e água no mancal?
  referência: A WSeal combina um anel V'Ring de duplo lábio com uma calota metálica, o que funciona como um labirinto e impede que poeira e água penetrem na parte interna do mancal.
  páginas [18] | seções ['7.2

## 10. Recuperação

Um chunk é **relevante por página** quando o intervalo de páginas dele toca uma página do gabarito, e **relevante de forma estrita** quando contém o `trecho_ancora`. Os chunks do `LEGADO` não trazem metadados: para medi-los, cada chunk recebe as páginas em que seus parágrafos aparecem no PDF.

Para cada pergunta dentro do escopo, `hit@k` diz se há algum chunk relevante entre os `k` primeiros; `precision@k` é a fração de relevantes entre os `k`; `MRR@k` é o inverso da posição do primeiro relevante; `context precision@k` é a precisão média nas posições dos relevantes (peso pela posição). `k` ∈ {1, 4}.

O grupo `com_estado` reúne só as perguntas que têm cenário de estado: é onde `NOVO sem estado` e `NOVO com estado` realmente diferem, porque nas demais as duas configurações recebem a mesma consulta.

### Calibração de `PESO_ESTADO` e `LIMIAR_ABSTENCAO` (somente `dev`)

A grade abaixo usa só as 5 perguntas `dev` dentro do escopo e as 5 fora do escopo. A escolha maximiza o hit@4 entre os pares que se abstêm em todas as perguntas fora do escopo; o desempate prefere menos abstenções indevidas nas perguntas dentro do escopo. O `teste` só é lido depois, nas tabelas seguintes.

In [9]:
import pandas as pd
from IPython.display import display

from src.pln.avaliacao import escolher_parametros, varrer_parametros
from src.pln.config import LIMIAR_ABSTENCAO, PESO_ESTADO

grade_dev = varrer_parametros(retriever, perguntas, estados)
display(pd.DataFrame(grade_dev))
escolha = escolher_parametros(grade_dev)
print(f"Par escolhido no dev: PESO_ESTADO={escolha['peso_estado']} | LIMIAR_ABSTENCAO={escolha['limiar_abstencao']}")
print(f"Par em config.py:     PESO_ESTADO={PESO_ESTADO} | LIMIAR_ABSTENCAO={LIMIAR_ABSTENCAO}")
assert (escolha["peso_estado"], escolha["limiar_abstencao"]) == (PESO_ESTADO, LIMIAR_ABSTENCAO), "config.py diverge da calibração"

,peso_estado,limiar_abstencao,hit4_dentro,n_dentro,abstencao_fora_correta,n_fora,abstencao_dentro_indevida
0,0.00,-4.0,5,5,0,5,0
1,0.00,-3.0,5,5,0,5,1
2,0.00,-2.0,5,5,1,5,1
3,0.00,-1.0,5,5,3,5,1
4,0.00,0.0,5,5,3,5,1
5,0.00,0.5,5,5,5,5,1
6,0.00,1.0,5,5,5,5,1
7,0.00,1.5,5,5,5,5,1
8,0.00,2.0,5,5,5,5,1
9,0.00,2.5,5,5,5,5,1


Par escolhido no dev: PESO_ESTADO=0.15 | LIMIAR_ABSTENCAO=2.0
Par em config.py:     PESO_ESTADO=0.15 | LIMIAR_ABSTENCAO=2.0


In [10]:
from src.pln.avaliacao import ConfiguracaoRecuperacao, executar_recuperacao, resumir_recuperacao
from src.pln.config import KS_AVALIACAO
from src.pln.rag import criar_retriever

retriever_legado = criar_retriever(chunks_legado, LEGADO)
configuracoes = [
    ConfiguracaoRecuperacao("LEGADO", retriever_legado, None, paginas),
    ConfiguracaoRecuperacao("NOVO sem estado", retriever, None),
    ConfiguracaoRecuperacao("NOVO com estado", retriever, estados),
]
registros_recuperacao = {c.nome: executar_recuperacao(c, perguntas, KS_AVALIACAO) for c in configuracoes}
resumos_recuperacao = {nome: resumir_recuperacao(regs, KS_AVALIACAO) for nome, regs in registros_recuperacao.items()}
tabela_recuperacao = pd.DataFrame(
    [{"configuracao": nome, **linha} for nome, linhas in resumos_recuperacao.items() for linha in linhas]
)
COLUNAS = ["configuracao", "grupo", "k", "n", "hit", "precision", "mrr", "context_precision"]


def mostrar_recuperacao(criterio, grupos):
    recorte = tabela_recuperacao[(tabela_recuperacao.criterio == criterio) & tabela_recuperacao.grupo.isin(grupos)]
    display(recorte.sort_values(["grupo", "k", "configuracao"])[COLUNAS].round(3).reset_index(drop=True))


print("Relevância por página — teste (15), dev (5) e perguntas com estado")
mostrar_recuperacao("pagina", ["teste", "dev", "com_estado"])
print("Relevância estrita (chunk contém a âncora) — teste, dev e perguntas com estado")
mostrar_recuperacao("estrita", ["teste", "dev", "com_estado"])
print("Relevância por página — teste, por categoria")
mostrar_recuperacao("pagina", [f"teste/{categoria}" for categoria in ("elétrica", "mecânica", "preventiva", "especificação")])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2688.72it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 3375.94it/s]

Relevância por página — teste (15), dev (5) e perguntas com estado


,configuracao,grupo,k,n,hit,precision,mrr,context_precision
0,LEGADO,com_estado,1,9,0.333,0.333,0.333,0.333
1,NOVO com estado,com_estado,1,9,0.889,0.889,0.889,0.889
2,NOVO sem estado,com_estado,1,9,0.889,0.889,0.889,0.889
3,LEGADO,com_estado,4,9,0.556,0.222,0.389,0.343
4,NOVO com estado,com_estado,4,9,1.000,0.500,0.944,0.846
5,NOVO sem estado,com_estado,4,9,1.000,0.472,0.944,0.917
6,LEGADO,dev,1,5,0.200,0.200,0.200,0.200
7,NOVO com estado,dev,1,5,0.800,0.800,0.800,0.800
8,NOVO sem estado,dev,1,5,0.800,0.800,0.800,0.800
9,LEGADO,dev,4,5,0.800,0.300,0.417,0.383


Relevância estrita (chunk contém a âncora) — teste, dev e perguntas com estado


,configuracao,grupo,k,n,hit,precision,mrr,context_precision
0,LEGADO,com_estado,1,9,0.222,0.222,0.222,0.222
1,NOVO com estado,com_estado,1,9,0.778,0.778,0.778,0.778
2,NOVO sem estado,com_estado,1,9,0.778,0.778,0.778,0.778
3,LEGADO,com_estado,4,9,0.444,0.111,0.287,0.287
4,NOVO com estado,com_estado,4,9,0.889,0.222,0.833,0.833
5,NOVO sem estado,com_estado,4,9,1.000,0.250,0.843,0.843
6,LEGADO,dev,1,5,0.200,0.200,0.200,0.200
7,NOVO com estado,dev,1,5,0.800,0.800,0.800,0.800
8,NOVO sem estado,dev,1,5,0.800,0.800,0.800,0.800
9,LEGADO,dev,4,5,0.200,0.050,0.200,0.200


Relevância por página — teste, por categoria


,configuracao,grupo,k,n,hit,precision,mrr,context_precision
0,LEGADO,teste/elétrica,1,4,0.500,0.500,0.500,0.500
1,NOVO com estado,teste/elétrica,1,4,1.000,1.000,1.000,1.000
2,NOVO sem estado,teste/elétrica,1,4,1.000,1.000,1.000,1.000
3,LEGADO,teste/elétrica,4,4,0.750,0.312,0.583,0.542
4,NOVO com estado,teste/elétrica,4,4,1.000,0.562,1.000,0.889
5,NOVO sem estado,teste/elétrica,4,4,1.000,0.625,1.000,0.903
6,LEGADO,teste/especificação,1,4,0.750,0.750,0.750,0.750
7,NOVO com estado,teste/especificação,1,4,1.000,1.000,1.000,1.000
8,NOVO sem estado,teste/especificação,1,4,1.000,1.000,1.000,1.000
9,LEGADO,teste/especificação,4,4,1.000,0.312,0.833,0.792


In [11]:
def valor(nome, grupo, criterio, k, metrica):
    linha = tabela_recuperacao[
        (tabela_recuperacao.configuracao == nome) & (tabela_recuperacao.grupo == grupo)
        & (tabela_recuperacao.criterio == criterio) & (tabela_recuperacao.k == k)
    ]
    return float(linha[metrica].iloc[0])


def comparar(melhor, base, grupo):
    for criterio in ("pagina", "estrita"):
        for metrica in ("hit", "mrr", "context_precision"):
            a, b = valor(melhor, grupo, criterio, 4, metrica), valor(base, grupo, criterio, 4, metrica)
            sentido = "supera" if a > b else "empata com" if a == b else "NÃO supera"
            print(f"[{grupo}/{criterio}] {metrica}@4: {melhor} {a:.3f} {sentido} {base} {b:.3f}")


comparar("NOVO sem estado", "LEGADO", "teste")
print()
comparar("NOVO com estado", "NOVO sem estado", "com_estado")

[teste/pagina] hit@4: NOVO sem estado 0.933 supera LEGADO 0.800
[teste/pagina] mrr@4: NOVO sem estado 0.933 supera LEGADO 0.628
[teste/pagina] context_precision@4: NOVO sem estado 0.896 supera LEGADO 0.606
[teste/estrita] hit@4: NOVO sem estado 0.933 supera LEGADO 0.400
[teste/estrita] mrr@4: NOVO sem estado 0.883 supera LEGADO 0.272
[teste/estrita] context_precision@4: NOVO sem estado 0.883 supera LEGADO 0.272

[com_estado/pagina] hit@4: NOVO com estado 1.000 empata com NOVO sem estado 1.000
[com_estado/pagina] mrr@4: NOVO com estado 0.944 empata com NOVO sem estado 0.944
[com_estado/pagina] context_precision@4: NOVO com estado 0.846 NÃO supera NOVO sem estado 0.917
[com_estado/estrita] hit@4: NOVO com estado 0.889 NÃO supera NOVO sem estado 1.000
[com_estado/estrita] mrr@4: NOVO com estado 0.833 NÃO supera NOVO sem estado 0.843
[com_estado/estrita] context_precision@4: NOVO com estado 0.833 NÃO supera NOVO sem estado 0.843


In [12]:
from src.pln.avaliacao import gravar_json
from src.pln.config import AVALIACAO_RECUPERACAO_OUTPUT_FILE

gravar_json(
    AVALIACAO_RECUPERACAO_OUTPUT_FILE,
    {
        "ks": list(KS_AVALIACAO),
        "parametros": {"peso_estado": PESO_ESTADO, "limiar_abstencao": LIMIAR_ABSTENCAO},
        "grade_dev": grade_dev,
        "configuracoes": {
            nome: {"resumo": resumos_recuperacao[nome], "registros": registros_recuperacao[nome]}
            for nome in registros_recuperacao
        },
    },
)
print(f"Gravado: {AVALIACAO_RECUPERACAO_OUTPUT_FILE.name}")

Gravado: avaliacao_recuperacao.json


## 11. Juiz por afirmação

A resposta é dividida em frases por código (sem LLM), ignorando a linha `Confiança:`, as citações `[p., seç.]` e a frase de abstenção. O juiz (`MODELO_JUIZ`, de família diferente do gerador, temperatura 0, com cache) recebe os chunks usados e uma afirmação por vez e devolve **um** rótulo:

| Rótulo | Significado |
|---|---|
| `SUSTENTADA` | decorre do contexto |
| `EXTERNA` | pode até ser verdadeira, mas **não está no contexto** |
| `CONTRADIZ` | contraria o contexto |
| `META` | não é afirmação factual (fora da conta) |

**faithfulness = SUSTENTADA / (SUSTENTADA + EXTERNA + CONTRADIZ)**. Duas guardas determinísticas prevalecem sobre o juiz: qualquer número da afirmação ausente do contexto, ou um marcador de conhecimento externo (`conhecimento geral`, `em geral`, `normalmente`… lista em `config.py`; ignorado se o próprio contexto usa a expressão), torna a afirmação `EXTERNA`. Resposta de abstenção sem conteúdo factual tem faithfulness 1,0, e a abstenção é avaliada à parte.

### Calibração

`data/juiz_calibracao.json` traz 16 casos `(contexto, afirmação, rótulo esperado)` escritos à mão: 5 `EXTERNA` (incluindo o `V-ring` e a "classe de isolamento F" quando o contexto não a traz), 4 `CONTRADIZ`, 5 `SUSTENTADA` e 2 `META`. Seis casos formam o `dev` da calibração; o prompt só pode ser ajustado (até 3 vezes) olhando esses seis. A versão do prompt abaixo é a primeira: acertou os 6 casos `dev`, então nenhum ajuste foi feito, e os 10 casos `teste` só foram lidos nesta execução. O resultado é reportado como saiu, nos dois modos: o juiz sozinho e o juiz com as guardas.

In [13]:
from IPython.display import Markdown

from src.pln.avaliacao import Juiz, calibrar_juiz, carregar_casos_calibracao
from src.pln.config import JUIZ_CALIBRACAO_OUTPUT_FILE, MODELO_GERADOR, MODELO_JUIZ, PROMPT_JUIZ_AFIRMACAO
from src.pln.llm import create_openrouter_client

assert MODELO_JUIZ != MODELO_GERADOR
cliente_llm = create_openrouter_client()
juiz = Juiz(cliente_llm)
display(Markdown(f"Modelo do juiz: `{MODELO_JUIZ}` | modelo do gerador: `{MODELO_GERADOR}`\n\n```text\n{PROMPT_JUIZ_AFIRMACAO}\n```"))

calibracao = calibrar_juiz(juiz, carregar_casos_calibracao())


def formatar(valor):
    return "n/d" if valor is None else f"{valor:.3f}"


for split in ("dev", "teste", "todos"):
    for modo in ("juiz_isolado", "juiz_com_guardas"):
        metricas = calibracao["metricas"][split][modo]
        print(f"{split:6} {modo:17} n={metricas['n']:2} acurácia={formatar(metricas['acuracia'])} "
              f"revocação EXTERNA={formatar(metricas['revocacao_externa'])} precisão EXTERNA={formatar(metricas['precisao_externa'])}")

print("\nMatriz de confusão (todos os casos; linhas = esperado, colunas = previsto)")
display(pd.DataFrame(calibracao["metricas"]["todos"]["juiz_isolado"]["matriz_confusao"]).T.rename_axis("juiz isolado"))
display(pd.DataFrame(calibracao["metricas"]["todos"]["juiz_com_guardas"]["matriz_confusao"]).T.rename_axis("juiz com guardas"))

divergentes = [caso for caso in calibracao["casos"] if caso["final"] != caso["esperado"]]
print(f"\nCasos em que o resultado final diverge do rótulo esperado: {len(divergentes)}")
for caso in divergentes:
    print(f"  {caso['id']} ({caso['split']}): esperado {caso['esperado']}, juiz {caso['juiz']}, final {caso['final']} — {caso['afirmacao']}")

gravar_json(JUIZ_CALIBRACAO_OUTPUT_FILE, {**calibracao, "prompt": PROMPT_JUIZ_AFIRMACAO, "ajustes_de_prompt_com_dev": 0})

Modelo do juiz: `google/gemma-3-12b-it` | modelo do gerador: `meta-llama/llama-3.3-70b-instruct`

```text
Você é um verificador rigoroso de fidelidade. Receberá um CONTEXTO (trechos de um manual) e uma AFIRMAÇÃO. Classifique a afirmação com UM rótulo, considerando SOMENTE o CONTEXTO: SUSTENTADA: o contexto afirma ou implica diretamente a afirmação. EXTERNA: a afirmação não está no contexto, mesmo que seja verdadeira ou de conhecimento comum; inclui função, causa, material, valor ou procedimento que o contexto não traz. CONTRADIZ: o contexto afirma algo incompatível com a afirmação. META: não é afirmação factual sobre o motor (saudação, aviso, organização da resposta). Na dúvida entre SUSTENTADA e EXTERNA, responda EXTERNA. Responda apenas com o rótulo.
```

dev    juiz_isolado      n= 6 acurácia=1.000 revocação EXTERNA=1.000 precisão EXTERNA=1.000


dev    juiz_com_guardas  n= 6 acurácia=1.000 revocação EXTERNA=1.000 precisão EXTERNA=1.000
teste  juiz_isolado      n=10 acurácia=0.900 revocação EXTERNA=1.000 precisão EXTERNA=0.750
teste  juiz_com_guardas  n=10 acurácia=0.900 revocação EXTERNA=1.000 precisão EXTERNA=0.750
todos  juiz_isolado      n=16 acurácia=0.938 revocação EXTERNA=1.000 precisão EXTERNA=0.833
todos  juiz_com_guardas  n=16 acurácia=0.938 revocação EXTERNA=1.000 precisão EXTERNA=0.833

Matriz de confusão (todos os casos; linhas = esperado, colunas = previsto)


,SUSTENTADA,EXTERNA,CONTRADIZ,META,INVALIDO
juiz isolado,,,,,
SUSTENTADA,5,0,0,0,0
EXTERNA,0,5,0,0,0
CONTRADIZ,0,1,3,0,0
META,0,0,0,2,0


,SUSTENTADA,EXTERNA,CONTRADIZ,META,INVALIDO
juiz com guardas,,,,,
SUSTENTADA,5,0,0,0,0
EXTERNA,0,5,0,0,0
CONTRADIZ,0,1,3,0,0
META,0,0,0,2,0



Casos em que o resultado final diverge do rótulo esperado: 1
  c10 (teste): esperado CONTRADIZ, juiz EXTERNA, final EXTERNA — Todas as carcaças dos motores W22 são fornecidas com grau de proteção IP55.


## 12. Respostas

O assistente responde as 25 perguntas com `MODELO_GERADOR`, na configuração `NOVO com estado` (a pergunta recebe o cenário de estado, quando tem um). A memória é reiniciada a cada pergunta, para que uma não herde a outra. Para cada resposta, o juiz rotula as afirmações (faithfulness) e compara a resposta com a referência (answer relevancy: `COBRE` = 1, `PARCIAL` = 0,5, `NAO_COBRE` = 0, mais o cosseno entre os embeddings de resposta e referência como segundo sinal).

Duas regras de abstenção: abstenção em pergunta **dentro** do escopo vale `NAO_COBRE`; resposta factual a pergunta **fora** do escopo é `alucinacao_fora_escopo`, nunca acerto. Falha persistente de API marca a pergunta como `erro`, sem trocar de modelo. Os `data/saida/avaliacao_*.json` guardam, por pergunta, modelo, consulta, chunks (página, seção, scores), resposta, afirmações rotuladas e métricas.

In [14]:
from src.pln.assistente import Assistente
from src.pln.avaliacao import ContextoRespostas, executar_respostas, resumir_respostas
from src.pln.config import AVALIACAO_RESPOSTAS_OUTPUT_FILE

assistente_avaliacao = Assistente(retriever, cliente_llm)
contexto_respostas = ContextoRespostas(assistente_avaliacao, juiz, retriever.componentes.embedding, estados)
registros_respostas = executar_respostas(contexto_respostas, perguntas)
resumo_respostas = resumir_respostas(registros_respostas)

linhas_respostas = [
    {
        "id": r["id"], "escopo": r["escopo"], "split": r["split"], "abstencao": r.get("abstencao"),
        "faithfulness": r.get("faithfulness"), "afirmações": r.get("contagem"),
        "cobertura": r.get("cobertura"), "cosseno": r.get("cosseno"),
        "alucinação": r.get("alucinacao_fora_escopo"), "erro": r.get("erro"),
    }
    for r in registros_respostas
]
display(pd.DataFrame(linhas_respostas).round(3))
display(pd.DataFrame({k: v for k, v in resumo_respostas.items() if k != "por_categoria"}).round(3))
display(pd.DataFrame(resumo_respostas["por_categoria"]).T.round(3))

O gerador citou página ou seção ausente dos chunks recuperados.


,id,escopo,split,abstencao,faithfulness,afirmações,cobertura,cosseno,alucinação,erro
0,e01,dentro,teste,False,1.0,"{'SUSTENTADA': 1, 'EXTERNA': 0, 'CONTRADIZ': 0...",COBRE,0.991,None,None
1,e02,dentro,dev,False,1.0,"{'SUSTENTADA': 1, 'EXTERNA': 0, 'CONTRADIZ': 0...",COBRE,0.976,None,None
2,e03,dentro,teste,False,1.0,"{'SUSTENTADA': 1, 'EXTERNA': 0, 'CONTRADIZ': 0...",COBRE,0.972,None,None
3,e04,dentro,teste,False,1.0,"{'SUSTENTADA': 1, 'EXTERNA': 0, 'CONTRADIZ': 0...",COBRE,0.971,None,None
4,e05,dentro,teste,False,1.0,"{'SUSTENTADA': 4, 'EXTERNA': 0, 'CONTRADIZ': 0...",PARCIAL,0.902,None,None
5,m01,dentro,teste,False,1.0,"{'SUSTENTADA': 1, 'EXTERNA': 0, 'CONTRADIZ': 0...",COBRE,0.979,None,None
6,m02,dentro,teste,False,1.0,"{'SUSTENTADA': 2, 'EXTERNA': 0, 'CONTRADIZ': 0...",COBRE,0.942,None,None
7,m03,dentro,dev,False,1.0,"{'SUSTENTADA': 1, 'EXTERNA': 0, 'CONTRADIZ': 0...",COBRE,0.966,None,None
8,m04,dentro,teste,False,1.0,"{'SUSTENTADA': 2, 'EXTERNA': 0, 'CONTRADIZ': 0...",COBRE,0.916,None,None
9,m05,dentro,dev,False,1.0,"{'SUSTENTADA': 1, 'EXTERNA': 0, 'CONTRADIZ': 0...",COBRE,0.975,None,None


,geral,dev,teste
n,25.000,10.000,15.000
erros,0.000,0.000,0.000
faithfulness_media,0.980,1.000,0.967
relevancy_media,0.850,0.800,0.867
cosseno_medio,0.959,0.969,0.956
abstencoes_indevidas,2.000,1.000,1.000
abstencoes_corretas,5.000,5.000,0.000
alucinacoes_fora_escopo,0.000,0.000,0.000


,n,erros,faithfulness_media,relevancy_media,cosseno_medio,abstencoes_indevidas,abstencoes_corretas,alucinacoes_fora_escopo
elétrica,5.0,0.0,1.0,0.9,0.962,0.0,0.0,0.0
mecânica,5.0,0.0,1.0,1.0,0.956,0.0,0.0,0.0
preventiva,5.0,0.0,1.0,0.6,0.966,2.0,0.0,0.0
especificação,5.0,0.0,0.9,0.9,0.954,0.0,0.0,0.0


In [15]:
geral = resumo_respostas["geral"]
print("Entrega anterior: faithfulness 4,75/5 e relevancy 3,50/5, nota 1–5 de um juiz único sobre a resposta inteira, sem context precision.")
print("Agora: faithfulness = fração de afirmações sustentadas (0–1) e relevancy em {0; 0,5; 1}, por afirmação e contra referência.")
print("O método mudou: os números não são comparáveis em escala; a conversão abaixo (nota/5) só dá ordem de grandeza.")
print(f"  anterior: faithfulness {4.75 / 5:.2f} | relevancy {3.50 / 5:.2f}")
print(f"  agora:    faithfulness {geral['faithfulness_media']:.2f} | relevancy {geral['relevancy_media']:.2f}")
print(f"Abstenções indevidas (dentro do escopo): {geral['abstencoes_indevidas']} | corretas (fora): {geral['abstencoes_corretas']}"
      f" | alucinações fora do escopo: {geral['alucinacoes_fora_escopo']} | erros: {geral['erros']}")

gravar_json(
    AVALIACAO_RESPOSTAS_OUTPUT_FILE,
    {"modelo_gerador": MODELO_GERADOR, "modelo_juiz": MODELO_JUIZ, "resumo": resumo_respostas, "registros": registros_respostas},
)
print(f"Gravado: {AVALIACAO_RESPOSTAS_OUTPUT_FILE.name}")

Entrega anterior: faithfulness 4,75/5 e relevancy 3,50/5, nota 1–5 de um juiz único sobre a resposta inteira, sem context precision.
Agora: faithfulness = fração de afirmações sustentadas (0–1) e relevancy em {0; 0,5; 1}, por afirmação e contra referência.
O método mudou: os números não são comparáveis em escala; a conversão abaixo (nota/5) só dá ordem de grandeza.
  anterior: faithfulness 0.95 | relevancy 0.70
  agora:    faithfulness 0.98 | relevancy 0.85
Abstenções indevidas (dentro do escopo): 2 | corretas (fora): 5 | alucinações fora do escopo: 0 | erros: 0
Gravado: avaliacao_respostas.json


## 13. Análise de erros

As cinco piores respostas pela média entre faithfulness e pontos de relevancy (pergunta fora do escopo vale 1 se houve abstenção e 0 se houve resposta factual). A **causa provável** é uma regra mecânica, não um julgamento: *recuperação* quando nenhum dos 4 chunks cai numa página do gabarito (ou o assistente se absteve); *geração* quando a recuperação acertou mas há afirmações `EXTERNA` ou `CONTRADIZ`; *gabarito ou juiz* quando a recuperação acertou, nada foi marcado como externo e ainda assim a cobertura não é total — caso para conferir a `resposta_referencia` e a resposta lado a lado.

In [16]:
recuperacao_por_id = {r["id"]: r for r in registros_recuperacao["NOVO com estado"]}
por_id = {pergunta.id: pergunta for pergunta in perguntas}


def nota(registro):
    if registro.get("erro"):
        return 0.0
    if registro["escopo"] == "fora":
        return 1.0 if registro["abstencao_correta"] else 0.0
    fidelidade = registro.get("faithfulness")
    return ((1.0 if fidelidade is None else fidelidade) + registro["pontos"]) / 2


def causa_provavel(registro):
    if registro.get("erro"):
        return "erro de API"
    if registro["escopo"] == "fora":
        return "geração (respondeu fora do escopo)"
    achou_pagina = recuperacao_por_id[registro["id"]]["metricas"]["pagina"]["4"]["hit"] == 1.0
    contagem = registro.get("contagem", {})
    if registro["abstencao"] or not achou_pagina:
        return "recuperação"
    if contagem.get("EXTERNA", 0) or contagem.get("CONTRADIZ", 0):
        return "geração"
    return "gabarito ou juiz"


piores = sorted(registros_respostas, key=nota)[:5]
for registro in piores:
    pergunta = por_id[registro["id"]]
    paginas_recuperadas = [chunk["pagina"] for chunk in registro.get("chunks", [])]
    print(f"[{registro['id']}] nota {nota(registro):.2f} | causa provável: {causa_provavel(registro)}")
    print(f"  pergunta: {pergunta.pergunta}")
    print(f"  páginas do gabarito {list(pergunta.paginas_gabarito)} × recuperadas {paginas_recuperadas}")
    print(f"  resposta: {registro.get('resposta', registro.get('mensagem_erro'))}")
    for afirmacao in registro.get("afirmacoes", []):
        print(f"    - {afirmacao['rotulo']:10} {afirmacao['afirmacao']}")
    print()

[p01] nota 0.50 | causa provável: recuperação
  pergunta: Que efeito a lubrificação em excesso pode ter no mancal?
  páginas do gabarito [14, 15] × recuperadas [16, 15, 20, 15]
  resposta: Não encontrei informações suficientes no manual do motor WEG W22 para responder a essa pergunta.
Confiança: Baixa

[p02] nota 0.50 | causa provável: recuperação
  pergunta: Qual é a recomendação para os bujões de dreno dos motores W22?
  páginas do gabarito [11] × recuperadas [23, 23, 14, 13]
  resposta: Não encontrei informações suficientes no manual do motor WEG W22 para responder a essa pergunta.
Confiança: Baixa

[s02] nota 0.50 | causa provável: geração
  pergunta: Qual é o fator de serviço do motor W22 Super Premium?
  páginas do gabarito [8] × recuperadas [8, 34, 19, 37]
  resposta: O fator de serviço do motor W22 Super Premium é 1,25 até a carcaça 315S/M [p. 8, seç. 1]. No entanto, em outra seção, é mencionado que o fator de serviço é aproximadamente 1,15 [p. 19, seç. 9]. Além disso, em [p. 3

## 14. Como os cenários são escolhidos

O feedback pede que a **anomalia elétrica**, a **anomalia mecânica** e a **manutenção preventiva** sejam demonstradas **separadamente**. Cada cenário parte de um episódio **real** dos alertas da Sprint 3 (nada de estado escrito à mão), escolhido por **regra e não à mão**, e a escolha fica registrada em `data/cenarios.json` para reprodução:

| Cenário | Regra (empate: o mais recente) |
|---|---|
| Elétrica | alerta de maior severidade (crítico) da grandeza *corrente elétrica* |
| Mecânica | alerta crítico de *vibração no mancal* (da *velocidade do eixo* se a vibração não tiver crítico) |
| Preventiva | instante sem alerta moderado ou crítico nas 24 h anteriores (operação normal ou só alerta leve) |
| Adversarial (seção 18) | alerta crítico de *velocidade do eixo* com o **menor** valor máximo, sozinho no estado: o valor que um modelo tende a achar normal |

Cada cenário tem **3 turnos**: (1) a pergunta natural do operador, (2) um **seguimento anafórico**, que exercita a memória (a consulta enviada ao retriever é a pergunta anterior + a atual) e (3) uma pergunta que **o catálogo não responde**, para mostrar a abstenção. Cada pergunta (1) e (2) traz a página e a seção do PDF que a sustentam (`fonte_esperada`, conferida no catálogo), comparadas com as fontes recuperadas. A célula abaixo reaplica as regras e confere que o registro de `data/cenarios.json` é o que elas devolvem.

In [17]:
import pandas as pd

from src.pln.cenarios import carregar_cenarios, descrever_episodio, escolher_episodio

conjunto = carregar_cenarios()
todos_cenarios = [*conjunto.cenarios, conjunto.adversarial]
episodios = {cenario.id: escolher_episodio(cenario.regra, alertas, leituras) for cenario in todos_cenarios}
tabela_episodios = pd.DataFrame(
    [
        {
            "cenário": cenario.id,
            "alerta": episodios[cenario.id].alerta_id or "—",
            "leitura": episodios[cenario.id].leitura_id,
            "instante (UTC)": episodios[cenario.id].instante.isoformat(timespec="seconds"),
            "grandeza": episodios[cenario.id].grandeza or "operação normal",
            "severidade": episodios[cenario.id].severidade or "—",
            "valor máx.": episodios[cenario.id].valor_max,
            "candidatos": episodios[cenario.id].candidatos,
        }
        for cenario in todos_cenarios
    ]
)
print(tabela_episodios.to_string(index=False))
for cenario in todos_cenarios:
    print(f"\n[{cenario.id}] {episodios[cenario.id].motivo}")
    assert descrever_episodio(episodios[cenario.id]) == dict(cenario.episodio), f"data/cenarios.json diverge da regra em {cenario.id}"


    cenário alerta                              leitura            instante (UTC)                         grandeza severidade  valor máx.  candidatos
   eletrica A-4468 2a22bbff-daaa-4f23-9b61-d79ba43ece03 2026-08-22T05:14:44+00:00       Corrente elétrica por fase    critico      172.01         537
   mecanica A-4471 b8d6892d-7340-4206-860a-8e780a48e431 2026-08-22T05:15:41+00:00 Vibração no mancal lado acoplado    critico        4.87         516
 preventiva      — 6c6a1a25-3898-4ce4-87fe-760f4bae7783 2026-08-23T09:46:39+00:00                  operação normal          —         NaN          34
adversarial A-1800 38490289-9cb3-4791-9330-52b433fd7a70 2026-05-07T20:38:05+00:00               Velocidade do eixo    critico     1850.47         115

[eletrica] 537 alertas critico de 'corrente'; o mais recente é A-4468.

[mecanica] 516 alertas critico de 'vibra'; o mais recente é A-4471.

[preventiva] 34 leituras sem alerta moderado/critico nas 24 h anteriores; a mais recente é a leitura 6c6a1a2

## 15. Cenário 1 — anomalia elétrica

O estado operacional vem do alerta de corrente escolhido na seção 14 (id do alerta e da leitura abaixo). O assistente é **novo** (memória própria) e responde os 3 turnos; cada resposta mostra a consulta usada, as fontes `[p., seç.]`, a confiança calculada, a abstenção e o modelo, e é avaliada pelo juiz por afirmação (faithfulness), pela cobertura da referência (relevancy) e pela conferência da `fonte_esperada` com as fontes recuperadas.

In [18]:
from IPython.display import display

from src.pln.assistente import Assistente
from src.pln.cenarios import AmbienteCenarios, executar_cenario, formatar_cenario, linhas_turnos

ambiente_cenarios = AmbienteCenarios(alertas, lambda: Assistente(retriever, cliente_llm), juiz, retriever.componentes.embedding)
resultado_eletrica = executar_cenario(conjunto.cenarios[0], ambiente_cenarios)
print(formatar_cenario(resultado_eletrica))
display(pd.DataFrame(linhas_turnos(resultado_eletrica)))


== Cenário 1 — anomalia elétrica ==
Episódio: alerta A-4468 | leitura 2a22bbff-daaa-4f23-9b61-d79ba43ece03 | 2026-08-22T05:14:44.244385+00:00
Regra: 537 alertas critico de 'corrente'; o mais recente é A-4468.
Estado: Teste (Sorocaba): severidade geral critico. Alerta A-4468: Corrente elétrica por fase, severidade critico, valor máximo 172.01 A, início 2026-08-22T05:14:44+00:00.

-- Turno 1 (natural): Qual sobrecorrente o motor W22 deve suportar e por quanto tempo?
   consulta usada: Qual sobrecorrente o motor W22 deve suportar e por quanto tempo?
   resposta: Conforme a norma ABNT NBR 17094, os motores com potência nominal até 315 kW e tensão nominal inferior ou igual a 1 kV devem ser capazes de suportar uma corrente igual a 1,5 vez a corrente nominal durante 2 minutos, no mínimo [p. 19, seç. 9]. 
Confiança: Média
   fontes: [p. 19, seç. 9] | confiança: Média | abstenção: False | modelo: meta-llama/llama-3.3-70b-instruct
   avaliação: faithfulness 1.0 | afirmações {'SUSTENTADA': 1, 'EX

,turno,tipo,consulta usada,fontes,fonte esperada recuperada,confiança,abstenção,faithfulness,cobertura,modelo
0,1,natural,Qual sobrecorrente o motor W22 deve suportar e...,"p. 19, seç. 9",True,Média,False,1.0,COBRE,meta-llama/llama-3.3-70b-instruct
1,2,seguimento,Qual sobrecorrente o motor W22 deve suportar e...,"p. 20, seç. 11.1",True,Média,False,0.5,PARCIAL,meta-llama/llama-3.3-70b-instruct
2,3,fora_escopo,Qual é a resistência ôhmica do enrolamento do ...,—,None,Baixa,True,1.0,NaN,meta-llama/llama-3.3-70b-instruct


## 16. Cenário 2 — anomalia mecânica

Mesmo protocolo, com o alerta de vibração no mancal escolhido na seção 14. O seguimento (turno 2) é anafórico ("desse mancal"): a consulta enviada ao retriever junta a pergunta anterior e a atual.

In [19]:
resultado_mecanica = executar_cenario(conjunto.cenarios[1], ambiente_cenarios)
print(formatar_cenario(resultado_mecanica))
display(pd.DataFrame(linhas_turnos(resultado_mecanica)))


== Cenário 2 — anomalia mecânica ==
Episódio: alerta A-4471 | leitura b8d6892d-7340-4206-860a-8e780a48e431 | 2026-08-22T05:15:41.161669+00:00
Regra: 516 alertas critico de 'vibra'; o mais recente é A-4471.
Estado: Teste (Sorocaba): severidade geral critico. Alerta A-4471: Vibração no mancal lado acoplado, severidade critico, valor máximo 4.87 mm/s, início 2026-08-22T05:15:41+00:00.

-- Turno 1 (natural): Com a vibração acima do limite, quais níveis de velocidade de vibração o catálogo admite para o motor W22?
   consulta usada: Com a vibração acima do limite, quais níveis de velocidade de vibração o catálogo admite para o motor W22?
   resposta: De acordo com o catálogo, os níveis de velocidade de vibração RMS em mm/s, medidos em condição de suspensão livre (base elástica) para os graus A e B são exibidos na tabela 5 [p. 14, seç. 5.2]. Para o grau A, os limites são: 1,6 mm/s para altura de eixo entre 56 e 132 mm, 2,2 mm/s para altura de eixo entre 132 e 280 mm, e 2,8 mm/s para altura d

,turno,tipo,consulta usada,fontes,fonte esperada recuperada,confiança,abstenção,faithfulness,cobertura,modelo
0,1,natural,"Com a vibração acima do limite, quais níveis d...","p. 14, seç. 5.2, p. 14, seç. 5.2",True,Média,False,1.0,COBRE,meta-llama/llama-3.3-70b-instruct
1,2,seguimento,"Com a vibração acima do limite, quais níveis d...","p. 14, seç. 5.2, p. 13, seç. 4.3, p. 14, seç. ...",True,Média,True,1.0,NAO_COBRE,meta-llama/llama-3.3-70b-instruct
2,3,fora_escopo,Qual é o torque de aperto dos parafusos da tam...,—,None,Baixa,True,1.0,NaN,meta-llama/llama-3.3-70b-instruct


## 17. Cenário 3 — manutenção preventiva

Aqui **não há alerta moderado ou crítico**: o estado é de operação normal (ou só alerta leve) e as perguntas são de procedimento de manutenção que o catálogo cobre. O instante é a leitura mais recente sem alerta moderado ou crítico nas 24 h anteriores.

In [20]:
resultado_preventiva = executar_cenario(conjunto.cenarios[2], ambiente_cenarios)
print(formatar_cenario(resultado_preventiva))
display(pd.DataFrame(linhas_turnos(resultado_preventiva)))


== Cenário 3 — manutenção preventiva ==
Episódio: alerta — | leitura 6c6a1a25-3898-4ce4-87fe-760f4bae7783 | 2026-08-23T09:46:39.901427+00:00
Regra: 34 leituras sem alerta moderado/critico nas 24 h anteriores; a mais recente é a leitura 6c6a1a25-3898-4ce4-87fe-760f4bae7783.
Estado: Teste (Sorocaba): operação normal em 2026-08-23T09:46:39+00:00.

-- Turno 1 (natural): Como lubrificar os rolamentos do motor W22?
   consulta usada: Como lubrificar os rolamentos do motor W22?
   resposta: Os motores W22 são fornecidos como padrão com pino graxeiro nas tampas dianteira e traseira para lubrificação dos rolamentos [p. 14, seç. 5.1]. Além disso, a vida útil do rolamento depende da correta utilização, manutenção e lubrificação, e ao respeitar a quantidade de graxa e os intervalos de lubrificação, permite-se que os rolamentos atinjam a vida útil descrita [p. 14, seç. 5.1]. Os intervalos de lubrificação sofrerão alterações em condições de operação diferentes de 40 °C de temperatura ambiente máxima

,turno,tipo,consulta usada,fontes,fonte esperada recuperada,confiança,abstenção,faithfulness,cobertura,modelo
0,1,natural,Como lubrificar os rolamentos do motor W22?,"p. 14, seç. 5.1, p. 14, seç. 5.1, p. 20, seç. ...",True,Média,False,1.0,COBRE,meta-llama/llama-3.3-70b-instruct
1,2,seguimento,Como lubrificar os rolamentos do motor W22? E ...,"p. 16, seç. 5.2.1",True,Média,False,1.0,COBRE,meta-llama/llama-3.3-70b-instruct
2,3,fora_escopo,Qual é a periodicidade recomendada para medir ...,—,None,Baixa,True,1.0,NaN,meta-llama/llama-3.3-70b-instruct


## 18. Limites e mitigações

### Cenário adversarial

Para observar se o assistente **respeita o estado** e **se abstém**, o estado é contraditório com a intuição de um modelo (alerta crítico de velocidade cujo valor está colado no nominal) e as perguntas induzem conhecimento externo. Os turnos adversariais não têm resposta no catálogo: o esperado é abstenção; resposta sem abster conta como "respondeu sem abster" na tabela de limites.

### Tabela de limites

Cada linha da tabela é montada **do que aconteceu** nas seções 10–18 (respostas da avaliação, cenários, calibração do juiz e dados) e cita o caso ou o número da execução em `evidencias`. A montagem falha se uma linha citar um id ou uma afirmação que a execução não produziu.

In [21]:
from src.pln.avaliacao import gravar_json
from src.pln.cenarios import consolidar_cenarios
from src.pln.config import CENARIOS_OUTPUT_FILE

resultado_adversarial = executar_cenario(conjunto.adversarial, ambiente_cenarios)
print(formatar_cenario(resultado_adversarial))
display(pd.DataFrame(linhas_turnos(resultado_adversarial)))

resultados_cenarios = [resultado_eletrica, resultado_mecanica, resultado_preventiva]
documento_cenarios = consolidar_cenarios(resultados_cenarios, resultado_adversarial)
gravar_json(CENARIOS_OUTPUT_FILE, documento_cenarios)
print(f"\nGravado: {CENARIOS_OUTPUT_FILE.name}")
display(pd.DataFrame(documento_cenarios["resumo"]).drop(index="confiancas").T)


== Cenário adversarial — estado contraditório e conhecimento externo ==
Episódio: alerta A-1800 | leitura 38490289-9cb3-4791-9330-52b433fd7a70 | 2026-05-07T20:38:05.538699+00:00
Regra: 115 alertas critico de 'velocidade', sozinhos no estado do instante, A-1800 tem o menor valor máximo (1850.47 rpm).
Estado: Teste (Sorocaba): severidade geral critico. Alerta A-1800: Velocidade do eixo, severidade critico, valor máximo 1850.47 rpm, início 2026-05-07T20:38:05+00:00.

-- Turno 1 (contradicao_estado): Os motores W22 de 4 polos giram a quantas rpm em 60 Hz? O valor do estado está normal?
   consulta usada: Os motores W22 de 4 polos giram a quantas rpm em 60 Hz? O valor do estado está normal?
   resposta: Os motores W22 de 4 polos giram a 1800rpm em 60 Hz [p. 22, seç. 12.1]. No entanto, o valor máximo de velocidade do eixo no estado operacional é de 1850,47 rpm, o que está acima do valor nominal para 4 polos em 60 Hz. Além disso, o alerta A-1800 indica uma severidade crítica para a velocidade

,turno,tipo,consulta usada,fontes,fonte esperada recuperada,confiança,abstenção,faithfulness,cobertura,modelo
0,1,contradicao_estado,Os motores W22 de 4 polos giram a quantas rpm ...,"p. 22, seç. 12.1, p. 22, seç. 12.1, p. 22, seç...",None,Média,False,0.0,None,meta-llama/llama-3.3-70b-instruct
1,2,conhecimento_externo,Qual é a causa raiz mais comum de vibração exc...,—,None,Baixa,True,1.0,None,meta-llama/llama-3.3-70b-instruct



Gravado: cenarios.json


,n_turnos,erros,faithfulness_media,relevancy_media,fontes_esperadas_recuperadas,fontes_esperadas,abstencoes_indevidas,abstencoes_corretas,respostas_sem_abster_fora_escopo
eletrica,3,0,0.833333,0.75,2,2,0,1,0
mecanica,3,0,1.0,0.5,2,2,1,1,0
preventiva,3,0,1.0,1.0,2,2,0,1,0
adversarial,2,0,0.5,None,0,0,0,1,1


In [22]:
from src.pln.cenarios import ExecucaoLimites, coletar_fatos_dados, montar_limites
from src.pln.config import LIMITES_OUTPUT_FILE

execucao_limites = ExecucaoLimites(
    registros_respostas, resultados_cenarios, resultado_adversarial, calibracao, coletar_fatos_dados(leituras)
)
limites = montar_limites(conjunto.modelos_limite, execucao_limites)
gravar_json(LIMITES_OUTPUT_FILE, limites)
print(f"Gravado: {LIMITES_OUTPUT_FILE.name} ({len(limites['linhas'])} linhas)")

pd.set_option("display.max_colwidth", None)
display(
    pd.DataFrame(
        [
            {
                "categoria": linha["categoria"],
                "limite observado": linha["limite"],
                "mitigação adotada": linha["mitigacao"],
                "risco que resta": linha["risco_residual"],
            }
            for linha in limites["linhas"]
        ]
    )
)


Gravado: limites.json (10 linhas)


,categoria,limite observado,mitigação adotada,risco que resta
0,fora_escopo,Perguntas fora do escopo do catálogo: 9 de 10 abstiveram e 1 responderam sem abster (ids: adversarial-t1).,"Abstenção determinística: sem chunk com escore bruto acima do limiar, o assistente devolve a frase fixa sem chamar o LLM.",O escore bruto não separa os escopos por construção: uma pergunta fora do escopo que recupere um chunk acima do limiar é respondida (1 de 10 nesta execução).
1,alucinacao,"Alucinação medida por afirmação: 6 de 42 afirmações rotuladas EXTERNA ou CONTRADIZ (6 EXTERNA, 0 CONTRADIZ; ids: adversarial-t1, eletrica-t2, s02).","Citação obrigatória [p., seç.] conferida contra os chunks, verificador numérico (número ausente do contexto vira EXTERNA), guarda de marcadores de conhecimento externo e juiz de família diferente da do gerador.","O juiz só vê os chunks e não o estado operacional: uma afirmação que vem do estado pode ser rotulada EXTERNA, e uma frase que o chunk não sustenta pode passar como SUSTENTADA."
2,alucinacao,"Estado contraditório com a intuição do modelo (Teste (Sorocaba): severidade geral critico. Alerta A-1800: Velocidade do eixo, severidade critico, valor máximo 1850.47 rpm, início 2026-05-07T20:38:05+00:00.): 1 de 2 turnos adversariais abstiveram.","Estado operacional tratado como dado confiável no prompt do sistema; severidade e confiança são calculadas por código, não pelo LLM.",Nada impede o LLM de redigir uma frase que minimize o alerta quando o catálogo traz uma informação parecida; só a leitura da resposta e o juiz por afirmação a detectam depois.
3,corpus,"O catálogo é comercial e não traz troubleshooting passo a passo: 3 de 3 perguntas de procedimento prático dos cenários abstiveram, e a fonte esperada foi recuperada em 6 de 6 turnos dentro do escopo.","Cenários só com perguntas que o PDF cobre (fonte esperada conferida na página) e uma pergunta deliberadamente sem resposta por cenário, para mostrar a abstenção.","Perguntas de diagnóstico (causa, reparo, torque) não têm resposta no corpus: o assistente só diz o que o catálogo afirma sobre o componente."
4,metodo,"Juiz LLM de 12B: acurácia 0,938 e revocação de EXTERNA 1,000 (precisão 0,833) em 16 casos de calibração; 1 caso(s) com rótulo final diferente do esperado.","Juiz diferente do gerador (famílias distintas, ids fixos, sem fallback), temperatura 0 com cache e guardas determinísticas que prevalecem sobre o rótulo do juiz.","Uma calibração de 16 casos não garante revocação em respostas reais: a faithfulness próxima de 1,0 pode esconder afirmações que o contexto não sustenta."
5,metodo,"Amostra pequena e gabarito por página: 20 perguntas dentro e 5 fora do escopo (0 fora no split teste), mais 11 turnos de cenário; faithfulness média 0,980.","Split dev/teste (calibração só no dev), relevância por página e estrita, e números reportados como saíram, inclusive as falhas.","Com 20 perguntas dentro do escopo, diferenças pequenas entre configurações não são distinguíveis; a abstenção correta só é medida nas perguntas do dev."
6,metodo,"Limiar fixo de abstenção (2,000 de escore bruto): 3 pergunta(s) dentro do escopo com abstenção indevida (ids: p01, p02, mecanica-t2).","Limiar calibrado só no dev, no par que abstém em todas as perguntas fora do escopo: prefere abster a responder sem sustentação.","O mesmo limiar abstém em perguntas legítimas cujo melhor chunk pontua abaixo de 2,000 (ids: p01, p02, mecanica-t2); o usuário recebe a frase de abstenção mesmo havendo resposta no catálogo."
7,dados,Só 4 das 6 variáveis têm limiar em active_assets.json; as outras 2 não geram alerta.,Variável sem limiar não gera alerta nem entra no estado; os limiares nunca são copiados à mão (sensores.py falha se divergirem de active_assets.json).,"Uma anomalia nas 2 variáveis sem limiar passa despercebida pelo estado operacional e, portanto, pelo assistente."
8,dados,Nomes e unidades de data_variables.json inconsistentes: 6 de 6 variáveis com unidade diferente do mapeamento canônico (e

In [23]:
for linha in limites["linhas"]:
    print(f"\n[{linha['categoria']}] {linha['id']}")
    for evidencia in linha["evidencias"]:
        citada = f" — \"{evidencia['afirmacao']}\"" if evidencia["afirmacao"] else ""
        print(f"  {evidencia['origem']}:{evidencia['id'] or '—'} {evidencia['detalhe']}{citada}")



[fora_escopo] fora_escopo_abstencao
  avaliacao_respostas:f01 abstenção; escore bruto 0,41; confiança Baixa
  avaliacao_respostas:f02 abstenção; escore bruto -2,70; confiança Baixa
  avaliacao_respostas:f03 abstenção; escore bruto -1,20; confiança Baixa
  avaliacao_respostas:f04 abstenção; escore bruto -1,90; confiança Baixa
  avaliacao_respostas:f05 abstenção; escore bruto 0,09; confiança Baixa
  cenarios:eletrica-t3 abstenção; escore bruto -1,90; confiança Baixa
  cenarios:mecanica-t3 abstenção; escore bruto -3,32; confiança Baixa
  cenarios:preventiva-t3 abstenção; escore bruto -4,20; confiança Baixa
  cenarios:adversarial-t1 respondeu sem abster; escore bruto 3,88; confiança Média
  cenarios:adversarial-t2 abstenção; escore bruto -5,64; confiança Baixa

[alucinacao] alucinacao_afirmacoes
  avaliacao_respostas:s02 EXTERNA (juiz) — "Além disso, em, é apresentado um exemplo de motor W22 IR3 Premium com fator de serviço 1,25, mas não se refere especificamente ao W22 Super Premium."
  

**Execução local:** rode as células em ordem; as seções 11 a 18 chamam o OpenRouter com a chave do `.env` da raiz. Não salve a chave no notebook.